<a href="https://colab.research.google.com/github/simbamufaz4-sudo/Actuarial_Science_Projects/blob/main/HASTS_201_Financial_Econometrics_Project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FINANCIAL ECONOMETRICS — Project #1
## Best-Practices Handbook: Volatility Modeling Challenges
### Apple Inc. (AAPL) · January 2018 – December 2025

---

**Topics Covered:**
1. Multicollinearity
2. Skewness
3. Sensitivity to Outliers
4. Overfitting

## Setup & Data Acquisition

In [ ]:
# ── Install / upgrade dependencies (run once in Colab)
!pip install yfinance --quiet
!pip install statsmodels --quiet
!pip install scikit-learn --quiet

In [ ]:
# ── Core imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import yfinance as yf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline

# ── Plot aesthetics
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
PALETTE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
print('Libraries loaded successfully.')

In [ ]:
# ── Download Apple Inc. (AAPL) data — 2018-01-01 to 2025-12-31
TICKER    = 'AAPL'
START     = '2018-01-01'
END       = '2025-12-31'

raw = yf.download(TICKER, start=START, end=END, auto_adjust=True)
raw.columns = raw.columns.get_level_values(0)          # flatten MultiIndex if present
raw.index   = pd.to_datetime(raw.index)

print(f'Downloaded {len(raw):,} trading days  ({raw.index[0].date()} → {raw.index[-1].date()})')
raw.head()

In [ ]:
# ── Feature engineering — build technical indicators used throughout the notebook
df = raw.copy()

# Returns
df['Return']       = df['Close'].pct_change()
df['Log_Return']   = np.log(df['Close'] / df['Close'].shift(1))

# Moving averages (intentionally correlated → multicollinearity demo)
df['SMA_5']        = df['Close'].rolling(5).mean()
df['SMA_10']       = df['Close'].rolling(10).mean()
df['SMA_20']       = df['Close'].rolling(20).mean()
df['EMA_10']       = df['Close'].ewm(span=10, adjust=False).mean()
df['EMA_20']       = df['Close'].ewm(span=20, adjust=False).mean()

# Volatility (rolling std)
df['Vol_5']        = df['Log_Return'].rolling(5).std()  * np.sqrt(252)
df['Vol_20']       = df['Log_Return'].rolling(20).std() * np.sqrt(252)

# Momentum
df['Momentum_5']   = df['Close'] - df['Close'].shift(5)
df['Momentum_10']  = df['Close'] - df['Close'].shift(10)

# Lagged returns
df['Lag1_Return']  = df['Return'].shift(1)
df['Lag2_Return']  = df['Return'].shift(2)

df.dropna(inplace=True)
print(f'Feature matrix shape: {df.shape}')
df[['Close','Return','Log_Return','SMA_5','SMA_20','Vol_20']].tail()

---
---
# CHALLENGE 1 — MULTICOLLINEARITY
---

## 1-A · Definition

Multicollinearity exists when two or more predictor variables in a regression model are **linearly dependent** (or nearly so). For predictor $X_j$, the degree of collinearity with the remaining predictors is quantified by the **Variance Inflation Factor (VIF)**:

$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$

where $R_j^2$ is the coefficient of determination from regressing $X_j$ on all other predictors. A complementary diagnostic is the **condition number** of the design matrix $\mathbf{X}$:

$$\kappa(\mathbf{X}) = \frac{\lambda_{\max}}{\lambda_{\min}}$$

where $\lambda_{\max}$ and $\lambda_{\min}$ are the largest and smallest singular values (eigenvalues) of $\mathbf{X}^\top\mathbf{X}$. High $\kappa$ signals ill-conditioning caused by collinearity.

## 1-B · Description

Multicollinearity occurs when predictor variables in a regression model carry redundant information — that is, one predictor can be closely approximated as a linear combination of others. In financial time-series modeling, technical indicators derived from the same price series (e.g., 5-day and 20-day simple moving averages) are classic sources of severe multicollinearity.

## 1-C · Demonstration

In [ ]:
# ── Regress AAPL Close price on five correlated moving-average / momentum features
feature_cols = ['SMA_5', 'SMA_10', 'SMA_20', 'EMA_10', 'EMA_20']
X_mc = sm.add_constant(df[feature_cols])
y_mc = df['Close']

model_mc = sm.OLS(y_mc, X_mc).fit()
print(model_mc.summary())

In [ ]:
# ── Compute VIF for each predictor
vif_data = pd.DataFrame({
    'Feature': feature_cols,
    'VIF'    : [variance_inflation_factor(X_mc.values, i+1)
                for i in range(len(feature_cols))]
})
vif_data['Severity'] = vif_data['VIF'].apply(
    lambda v: 'None' if v < 5 else ('Moderate' if v < 10 else 'Severe'))
print(vif_data.to_string(index=False))

# ── Condition number
_, s, _ = np.linalg.svd(X_mc.values)
cond_num = s.max() / s.min()
print(f'\nCondition Number κ = {cond_num:,.1f}  (>30 → severe collinearity)')

## 1-D · Diagram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Correlation heat-map
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=axes[0], mask=mask,
            linewidths=0.5, annot_kws={'size': 10})
axes[0].set_title('Pairwise Correlation Matrix\n(Moving-Average Features)', fontweight='bold')

# ── Right: VIF bar chart
colors = ['#d62728' if v >= 10 else '#ff7f0e' if v >= 5 else '#2ca02c'
          for v in vif_data['VIF']]
axes[1].barh(vif_data['Feature'], vif_data['VIF'], color=colors, edgecolor='k', linewidth=0.7)
axes[1].axvline(10, color='red',    linestyle='--', lw=1.5, label='VIF = 10 (Severe)')
axes[1].axvline(5,  color='orange', linestyle='--', lw=1.5, label='VIF = 5 (Moderate)')
axes[1].set_xlabel('Variance Inflation Factor (VIF)')
axes[1].set_title('VIF Scores — AAPL Technical Indicators', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('multicollinearity_diagram.png', bbox_inches='tight')
plt.show()
print('Figure saved.')

## 1-E · Diagnosis

| Test | Rule of Thumb | Interpretation |
|------|--------------|----------------|
| Pairwise correlation | $|r_{ij}| > 0.80$ | Potential collinearity |
| VIF | $> 5$ moderate · $> 10$ severe | Predictor $j$ explained by others |
| Condition number $\kappa$ | $> 30$ concerning · $> 100$ severe | Overall design matrix ill-conditioned |
| Sign of coefficient | Opposite to economic logic | Coefficients distorted by collinearity |

In [ ]:
# ── Diagnosis summary
print('=== MULTICOLLINEARITY DIAGNOSIS — AAPL ===')
print(f"Max pairwise correlation : {corr.where(mask==False).stack().abs().max():.4f}")
print(f"Max VIF                  : {vif_data['VIF'].max():,.1f}")
print(f"Condition Number         : {cond_num:,.1f}")
print('\nVerdict: ALL three indicators confirm SEVERE multicollinearity.')

## 1-F · Damage

**Multicollinearity does not bias $\hat{\boldsymbol{\beta}}$, but it inflates the variance of estimates**, rendering individual coefficients statistically unreliable. Concretely:

- **Standard errors balloon** → $t$-statistics collapse → predictors that *are* significant appear insignificant (Type II errors).
- **Signs flip unpredictably**: SMA_5 may show a *negative* coefficient even though rising short-term prices should increase the forecast.
- **Unstable estimates**: Adding or removing one correlated predictor radically changes all other coefficients, destroying model interpretability — a critical failure when derivatives desk traders rely on stable Greeks and risk factors.
- **Hedging errors**: Mis-estimated factor loadings propagate directly into delta and vega hedges, creating unquantified residual exposure.

## 1-G · Directions

In [ ]:
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

# ── Direction 1: Principal Component Analysis (PCA)
pca = PCA()
pca.fit(X_scaled)
explained = pd.Series(pca.explained_variance_ratio_,
                       index=[f'PC{i+1}' for i in range(len(feature_cols))])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA scree plot
axes[0].bar(explained.index, explained.values * 100, color=PALETTE, edgecolor='k', linewidth=0.6)
axes[0].plot(explained.index, explained.cumsum() * 100, 'ro--', label='Cumulative')
axes[0].axhline(95, color='gray', linestyle=':', label='95% threshold')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('PCA Scree Plot — Directions (Remedy)', fontweight='bold')
axes[0].legend()

# ── Direction 2: Ridge Regression (L2 regularization)
alphas = np.logspace(-2, 4, 100)
ridge_coefs = []
for a in alphas:
    ridge = Ridge(alpha=a, fit_intercept=True)
    ridge.fit(X_scaled, df['Close'].values)
    ridge_coefs.append(ridge.coef_)
ridge_coefs = np.array(ridge_coefs)

for i, feat in enumerate(feature_cols):
    axes[1].plot(np.log10(alphas), ridge_coefs[:, i], label=feat)
axes[1].set_xlabel('log₁₀(Ridge Penalty α)')
axes[1].set_ylabel('Standardized Coefficient')
axes[1].set_title('Ridge Regression Coefficient Path\n(Shrinks Collinear Coefficients)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].axhline(0, color='k', linewidth=0.8)

plt.tight_layout()
plt.savefig('multicollinearity_remedies.png', bbox_inches='tight')
plt.show()

print('\n── Recommended Directions ──')
print('1. PCA         : Collapse correlated features into orthogonal principal components.')
print('2. Ridge (L2)  : Add L2 penalty to shrink collinear coefficients toward zero.')
print('3. Lasso (L1)  : Automatic variable selection — drives redundant predictors to exactly 0.')
print('4. Feature drop: Retain only the most economically interpretable indicator (e.g., SMA_20).')

---
## 1 · Non-Technical Report — Multicollinearity

**What the data showed us:**
When building a model to forecast Apple's stock behavior, several of our inputs were
constructed from the same underlying price data — meaning they were essentially
measuring the same thing from slightly different angles. The model became confused
because it could not distinguish the individual contribution of each input, causing its
internal estimates to become unstable and contradictory.

**Recommended course of action:**
Rather than feeding the model many overlapping signals, the team should consolidate
them into a smaller set of independent, non-redundant inputs before modeling begins.
An additional safeguard is to apply a mathematical penalty during model fitting that
naturally suppresses inputs carrying duplicate information, preventing any single
redundant signal from distorting the final forecast.

**Factors that impact the portfolio:**
The most consequential downstream effect is on the accuracy of hedging ratios. When
the model's sensitivity estimates are unreliable, the number of contracts or shares
needed to neutralize a position is wrong — leading to either over-hedging (wasted
capital) or under-hedging (unquantified loss exposure). For a derivatives desk, this
translates directly to P&L volatility that was not budgeted for.

---

---
---
# CHALLENGE 2 — SKEWNESS
---

## 2-A · Definition

**Skewness** is the standardized third central moment of a distribution. For a sample $\{r_t\}_{t=1}^{T}$:

$$\gamma_1 = \frac{\frac{1}{T}\sum_{t=1}^{T}(r_t - \bar{r})^3}{\left[\frac{1}{T}\sum_{t=1}^{T}(r_t - \bar{r})^2\right]^{3/2}} = \frac{\mu_3}{\sigma^3}$$

- $\gamma_1 > 0$: right (positive) skew — long right tail.
- $\gamma_1 < 0$: left (negative) skew — long left tail, common in equity returns.
- $\gamma_1 = 0$: symmetric distribution (e.g., Gaussian).

The **Jarque–Bera test** jointly tests for zero skewness and zero excess kurtosis:

$$JB = \frac{T}{6}\left(\gamma_1^2 + \frac{(\gamma_2)^2}{4}\right) \xrightarrow{d} \chi^2_2$$

where $\gamma_2 = \mu_4/\sigma^4 - 3$ is excess kurtosis.

## 2-B · Description

Skewness measures the asymmetry of a return distribution relative to its mean — in equity markets, returns typically exhibit *negative* skewness because crashes are faster and more extreme than rallies. Ignoring skewness in volatility modeling leads to systematically mis-priced options (particularly puts) and underestimated tail risk in Value-at-Risk frameworks.

## 2-C · Demonstration

In [ ]:
returns = df['Log_Return'].dropna()

# ── Summary statistics
skew_val  = float(returns.skew())
kurt_val  = float(returns.kurt())      # excess kurtosis
jb_stat, jb_pval, _, _ = jarque_bera(returns)

# ── Compare with Gaussian of same mean/std
mu_r, sig_r = returns.mean(), returns.std()
normal_draws = np.random.normal(mu_r, sig_r, len(returns))

print('=== AAPL Log-Return Distributional Statistics ===')
stats_df = pd.DataFrame({
    'Statistic': ['Mean', 'Std Dev', 'Skewness (γ₁)', 'Excess Kurtosis (γ₂)',
                  'JB Statistic', 'JB p-value'],
    'AAPL Returns': [f'{mu_r:.6f}', f'{sig_r:.6f}', f'{skew_val:.4f}',
                     f'{kurt_val:.4f}', f'{jb_stat:.2f}', f'{jb_pval:.2e}'],
    'Normal Reference': ['0 (centered)', sig_r, '0.000', '0.000', '—', '—']
})
print(stats_df.to_string(index=False))

# ── 5th percentile (left tail)
p5_actual  = np.percentile(returns, 5)
p5_normal  = stats.norm.ppf(0.05, mu_r, sig_r)
print(f'\n5th Percentile (Actual) : {p5_actual:.4f}')
print(f'5th Percentile (Normal) : {p5_normal:.4f}')
print(f'→ True left tail is {abs(p5_actual/p5_normal - 1)*100:.1f}% fatter than Normal.')

## 2-D · Diagram

In [ ]:
fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])

# ── Panel 1: Histogram + Normal overlay
x_range = np.linspace(returns.min(), returns.max(), 300)
ax1.hist(returns, bins=80, density=True, color='steelblue',
         alpha=0.65, edgecolor='white', linewidth=0.3, label='AAPL Returns')
ax1.plot(x_range, stats.norm.pdf(x_range, mu_r, sig_r),
         'r-', lw=2, label='Normal Fit')
ax1.axvline(mu_r, color='k', linestyle='--', lw=1.2, label=f'Mean={mu_r:.4f}')
ax1.set_xlabel('Log Return')
ax1.set_ylabel('Density')
ax1.set_title(f'Return Distribution\n(Skewness = {skew_val:.4f})', fontweight='bold')
ax1.legend(fontsize=9)

# ── Panel 2: Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(returns, dist='norm')
ax2.scatter(osm, osr, s=5, alpha=0.4, color='steelblue', label='AAPL')
ax2.plot(osm, slope * np.array(osm) + intercept, 'r-', lw=2, label='Normal Line')
ax2.set_xlabel('Theoretical Quantiles')
ax2.set_ylabel('Sample Quantiles')
ax2.set_title('Q-Q Plot vs. Normal\n(Tail Deviations = Evidence of Skewness)', fontweight='bold')
ax2.legend(fontsize=9)

# ── Panel 3: Rolling skewness over time
rolling_skew = returns.rolling(63).skew()   # ~1 quarter
ax3.plot(rolling_skew.index, rolling_skew, color='darkorange', linewidth=0.9)
ax3.axhline(0, color='k', linestyle='--', lw=1)
ax3.fill_between(rolling_skew.index, rolling_skew, 0,
                 where=(rolling_skew < 0), color='red',    alpha=0.25, label='Neg. Skew')
ax3.fill_between(rolling_skew.index, rolling_skew, 0,
                 where=(rolling_skew >= 0), color='green', alpha=0.25, label='Pos. Skew')
ax3.set_xlabel('Date')
ax3.set_ylabel('Rolling Skewness (63-day)')
ax3.set_title('Time-Varying Skewness — AAPL', fontweight='bold')
ax3.legend(fontsize=9)

plt.savefig('skewness_diagram.png', bbox_inches='tight')
plt.show()

## 2-E · Diagnosis

In [ ]:
# ── Diagnosis tests
print('=== SKEWNESS DIAGNOSIS ===')
print(f'1. Sample skewness γ₁        = {skew_val:.4f}  (< 0 → left-skewed)')
print(f'2. Excess kurtosis γ₂         = {kurt_val:.4f}  (> 0 → fat tails)')
print(f'3. Jarque-Bera statistic       = {jb_stat:.2f}')
print(f'   JB p-value                  = {jb_pval:.2e}  → Reject normality at any conventional level')

# Shapiro-Wilk on a subsample
sw_stat, sw_pval = stats.shapiro(returns.sample(500, random_state=42))
print(f'4. Shapiro-Wilk (n=500 sample) = {sw_stat:.4f}, p = {sw_pval:.2e}')

print('\n── Rule of thumb for skewness severity ──')
print('|γ₁| < 0.5  : Approximately symmetric')
print('0.5 ≤ |γ₁| < 1  : Moderately skewed')
print('|γ₁| ≥ 1.0  : Highly skewed')
print(f'\nAAPL γ₁ = {skew_val:.4f} → {"Approximately symmetric" if abs(skew_val)<0.5 else "Moderately skewed" if abs(skew_val)<1 else "Highly skewed"}')

## 2-F · Damage

Ignoring skewness in financial time-series modeling produces **systematic errors across the derivatives desk**:

- **Option mispricing**: Black–Scholes assumes log-normal (zero-skew) returns. Negative skewness means puts are *underpriced* and calls *overpriced* by a symmetric model — the firm sells insurance too cheaply.
- **VaR underestimation**: A Gaussian VaR at 99% assumes the left tail is thin; negative skew pushes real losses beyond that threshold more frequently than modeled, violating Basel regulatory requirements.
- **GARCH misspecification**: Standard GARCH treats positive and negative shocks symmetrically. Negative skewness reflects the **leverage effect** (losses increase future volatility more than gains), requiring asymmetric specifications.
- **Reward–Risk distortion**: Portfolios with negatively-skewed assets appear safer on mean-variance metrics (Sharpe ratio) than they truly are, leading to excessive position sizes.

## 2-G · Directions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Direction 1: Skewed-t distribution fit
x_grid = np.linspace(returns.min(), returns.max(), 300)

# Fit Normal
norm_params = stats.norm.fit(returns)
# Fit Student-t
t_params = stats.t.fit(returns)

axes[0].hist(returns, bins=80, density=True, color='steelblue',
             alpha=0.5, label='AAPL Returns', edgecolor='white', lw=0.3)
axes[0].plot(x_grid, stats.norm.pdf(x_grid, *norm_params),
             'r-', lw=2, label='Normal')
axes[0].plot(x_grid, stats.t.pdf(x_grid, *t_params),
             'g--', lw=2, label="Student-t (fat tails)")
axes[0].set_xlabel('Log Return')
axes[0].set_ylabel('Density')
axes[0].set_title('Direction 1: Student-t vs Normal Fit\n(Student-t better captures skew/tail)', fontweight='bold')
axes[0].legend(fontsize=9)

# ── Direction 2: Box-Cox / log transformation to reduce skewness
positive_prices = df['Close'].values   # prices are always positive
skew_raw = stats.skew(positive_prices)
log_prices = np.log(positive_prices)
skew_log  = stats.skew(log_prices)

axes[1].hist(positive_prices, bins=50, density=True, alpha=0.5,
             color='salmon', label=f'Raw Price (skew={skew_raw:.2f})', edgecolor='white')
ax_twin = axes[1].twinx()
ax_twin.hist(log_prices, bins=50, density=True, alpha=0.5,
             color='steelblue', label=f'Log Price (skew={skew_log:.2f})', edgecolor='white')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Density (raw price)', color='salmon')
ax_twin.set_ylabel('Density (log price)', color='steelblue')
axes[1].set_title('Direction 2: Log-Transformation\n(Reduces Price Skewness)', fontweight='bold')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.savefig('skewness_remedies.png', bbox_inches='tight')
plt.show()

print('\n── Recommended Directions ──')
print('1. GJR-GARCH / EGARCH  : Asymmetric volatility models capturing the leverage effect.')
print('2. Skewed-t / NIG      : Replace Gaussian innovations with skewed fat-tailed distributions.')
print('3. Log-transformation  : Normalize price data before regression.')
print('4. Cornish-Fisher VaR  : Adjust quantile estimates for skewness and kurtosis.')

---
## 2 · Non-Technical Report — Skewness

**What the data showed us:**
Apple's daily return distribution is not symmetric. Large losses occur more frequently
and are more extreme than large gains of the same magnitude. In practical terms, the
downside of holding AAPL is materially worse than a standard bell-curve assumption
would suggest — particularly during stress periods like 2020 and 2022.

**Recommended course of action:**
Models used for pricing and risk management should be calibrated to explicitly allow
for this left-sided imbalance. This means using a distribution assumption that places
more probability weight on extreme negative outcomes, and updating that calibration
periodically since the degree of imbalance changes over time (as shown in the
rolling analysis).

**Factors that impact the portfolio:**
The primary risk is systematic underpricing of downside protection (put options) and
overpricing of upside participation (call options). If the desk sells puts at prices
derived from a symmetric model, it is collecting insufficient premium for the true
level of risk being taken on. In aggregate, this creates a slow, steady bleed of
risk-adjusted P&L that only becomes visible during drawdown episodes — exactly
when liquidity is tightest.

---

---
---
# CHALLENGE 3 — SENSITIVITY TO OUTLIERS
---

## 3-A · Definition

An **outlier** is an observation $y_t$ that lies far from the bulk of the data. In regression, its influence is measured by **Cook's Distance**:

$$D_i = \frac{(\hat{\boldsymbol{\beta}} - \hat{\boldsymbol{\beta}}_{(-i)})^\top (\mathbf{X}^\top\mathbf{X})(\hat{\boldsymbol{\beta}} - \hat{\boldsymbol{\beta}}_{(-i)})}{p \cdot \widehat{\sigma}^2}$$

where $\hat{\boldsymbol{\beta}}_{(-i)}$ is the OLS estimate with observation $i$ removed, and $p$ is the number of parameters. Equivalently:

$$D_i = \frac{\hat{e}_i^2}{p \cdot \text{MSE}} \cdot \frac{h_{ii}}{(1 - h_{ii})^2}$$

where $h_{ii}$ are the diagonal elements of the hat matrix $\mathbf{H} = \mathbf{X}(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top$ (leverage scores). A **standardized residual** $|e_i^*| > 3$ also flags outliers.

## 3-B · Description

Sensitivity to outliers refers to the disproportionate influence that a small number of extreme observations exert on OLS parameter estimates — in financial data, market crashes (e.g., COVID-19 in March 2020) and flash crashes create high-leverage outliers that can fundamentally distort a fitted model. Without robust treatment, a single crisis week can override the signal embedded in years of normal market behavior.

## 3-C · Demonstration

In [ ]:
# ── Regress today's return on yesterday's return
X_out = sm.add_constant(df[['Lag1_Return', 'Lag2_Return']])
y_out = df['Return']

model_full = sm.OLS(y_out, X_out).fit()

# ── Standardized residuals & leverage
influence  = model_full.get_influence()
std_resid  = influence.resid_studentized_internal
leverage   = influence.hat_matrix_diag
cooks_d    = influence.cooks_distance[0]

# ── Identify the COVID crash window: Feb-April 2020
covid_mask = (df.index >= '2020-02-01') & (df.index <= '2020-04-30')
outlier_mask = (np.abs(std_resid) > 3) | (cooks_d > 4 / len(df))

print(f'Total observations       : {len(df):,}')
print(f'Outliers |e*| > 3        : {(np.abs(std_resid) > 3).sum()}')
print(f"Outliers Cook's D > 4/n  : {(cooks_d > 4/len(df)).sum()}")
print(f'Dates flagged as outliers (sample):')
outlier_dates = df.index[outlier_mask]
print(outlier_dates[:10].strftime('%Y-%m-%d').tolist())

# ── Effect on coefficients: with vs. without outliers
X_clean = X_out[~outlier_mask]
y_clean = y_out[~outlier_mask]
model_clean = sm.OLS(y_clean, X_clean).fit()

coef_compare = pd.DataFrame({
    'With Outliers'    : model_full.params,
    'Without Outliers' : model_clean.params
})
coef_compare['% Change'] = ((coef_compare['Without Outliers'] - coef_compare['With Outliers'])
                             / coef_compare['With Outliers'].abs() * 100).round(2)
print('\n── Coefficient Comparison ──')
print(coef_compare)

## 3-D · Diagram

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dates = df.index

# ── Panel 1: AAPL returns with outliers highlighted
axes[0,0].plot(dates, y_out, color='steelblue', linewidth=0.6, label='Daily Return')
axes[0,0].scatter(dates[outlier_mask], y_out[outlier_mask],
                  color='red', s=30, zorder=5, label='Outlier (|e*|>3 or Cook\'s D>4/n)')
axes[0,0].axhline(0, color='k', linewidth=0.8)
axes[0,0].set_title('AAPL Daily Returns — Outliers Flagged', fontweight='bold')
axes[0,0].set_ylabel('Return')
axes[0,0].legend(fontsize=9)

# ── Panel 2: Cook's Distance
axes[0,1].stem(dates, cooks_d, linefmt='steelblue', markerfmt=' ', basefmt='k-')
threshold = 4 / len(df)
axes[0,1].axhline(threshold, color='red', linestyle='--', label=f"Threshold = 4/n = {threshold:.4f}")
axes[0,1].set_title("Cook's Distance — Influential Observations", fontweight='bold')
axes[0,1].set_ylabel("Cook's Distance")
axes[0,1].legend(fontsize=9)

# ── Panel 3: Leverage vs Standardized Residuals (influence plot)
sc = axes[1,0].scatter(leverage, std_resid,
                        c=cooks_d, cmap='hot_r', s=20, alpha=0.7)
axes[1,0].axhline(3,  color='red', linestyle='--', lw=1)
axes[1,0].axhline(-3, color='red', linestyle='--', lw=1)
axes[1,0].set_xlabel('Leverage $h_{ii}$')
axes[1,0].set_ylabel('Studentized Residual $e^*_i$')
axes[1,0].set_title("Leverage vs. Residual Plot\n(Color = Cook's D)", fontweight='bold')
plt.colorbar(sc, ax=axes[1,0], label="Cook's Distance")

# ── Panel 4: Coefficient stability
feat_labels = ['Intercept', 'Lag1_Return', 'Lag2_Return']
x_pos = np.arange(len(feat_labels))
width = 0.35
axes[1,1].bar(x_pos - width/2, model_full.params,  width, label='With Outliers',    color='#d62728', edgecolor='k')
axes[1,1].bar(x_pos + width/2, model_clean.params, width, label='Without Outliers', color='#1f77b4', edgecolor='k')
axes[1,1].set_xticks(x_pos)
axes[1,1].set_xticklabels(feat_labels)
axes[1,1].set_ylabel('Coefficient Value')
axes[1,1].set_title('Coefficient Shift\n(Impact of Outlier Removal)', fontweight='bold')
axes[1,1].legend(fontsize=9)
axes[1,1].axhline(0, color='k', linewidth=0.8)

plt.tight_layout()
plt.savefig('outlier_diagram.png', bbox_inches='tight')
plt.show()

## 3-E · Diagnosis

In [ ]:
print("=== OUTLIER / HIGH-INFLUENCE DIAGNOSIS ===")
print()
print("Test 1 — Standardized Residuals")
print(f"  |e*_i| > 3: {(np.abs(std_resid) > 3).sum()} observations flagged")
print()
print("Test 2 — Cook's Distance (threshold = 4/n)")
print(f"  D_i > {4/len(df):.4f}: {(cooks_d > 4/len(df)).sum()} observations flagged")
print()
print("Test 3 — Leverage (h_ii threshold = 2p/n)")
p = X_out.shape[1]
n = len(X_out)
lev_thresh = 2 * p / n
print(f"  h_ii > {lev_thresh:.4f}: {(leverage > lev_thresh).sum()} observations flagged")
print()
print("Test 4 — Z-score (for quick univariate check)")
z = np.abs(stats.zscore(y_out))
print(f"  |z| > 3: {(z > 3).sum()} return observations")

## 3-F · Damage

When OLS is applied to financial data containing extreme observations:

- **Coefficient bias**: The squared-error loss function gives outliers influence proportional to $e_i^2$, so a single crash day can dominate and drag all coefficients toward fitting that one observation.
- **Volatility overestimation during crises, underestimation otherwise**: A model fit including March 2020 systematically overestimates normal-market volatility and mis-calibrates option premiums.
- **Hedging instability**: Delta and vega computed from a contaminated model fluctuate erratically, requiring excessive rebalancing and incurring transaction costs.
- **Backtesting distortion**: In-sample statistics (R², RMSE) look artificially poor, causing the quant team to discard valid models or add unnecessary complexity.
- **Regulatory risk**: Underestimated tail risk from an outlier-sensitive VaR model can trigger Basel III capital shortfalls during the next stress event.

## 3-G · Directions

In [ ]:
from statsmodels.robust.robust_linear_model import RLM
import statsmodels.robust.norms as rn

# ── Direction 1: Huber Robust Regression (M-estimator)
model_huber = RLM(y_out, X_out, M=rn.HuberT()).fit()

# ── Direction 2: Winsorization (clip at 1st–99th percentile)
y_wins = y_out.clip(lower=y_out.quantile(0.01), upper=y_out.quantile(0.99))
model_wins = sm.OLS(y_wins, X_out).fit()

# ── Compare intercepts & slopes
methods = ['OLS (full)', 'OLS (excl. outliers)', 'Huber M-est.', 'OLS Winsorized']
lag1_coefs = [
    model_full.params['Lag1_Return'],
    model_clean.params['Lag1_Return'],
    model_huber.params['Lag1_Return'],
    model_wins.params['Lag1_Return']
]

fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = ['#d62728','#ff7f0e','#2ca02c','#1f77b4']
bars = ax.barh(methods, lag1_coefs, color=colors_bar, edgecolor='k', linewidth=0.7)
ax.axvline(0, color='k', linewidth=0.9)
ax.set_xlabel('Lag1_Return Coefficient')
ax.set_title('Outlier-Remedy Comparison: Lag1 Coefficient Across Methods', fontweight='bold')
for bar, val in zip(bars, lag1_coefs):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{val:.5f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('outlier_remedies.png', bbox_inches='tight')
plt.show()

print('\n── Recommended Directions ──')
print('1. Huber M-estimator   : Down-weights residuals exceeding a threshold c (default 1.345σ).')
print('2. LAD Regression      : Minimizes |e| instead of e² — median-based, fully robust.')
print('3. Winsorization       : Cap extreme returns at [1st, 99th] percentile before fitting.')
print('4. GARCH with jumps    : Explicitly model rare large shocks (Merton Jump-Diffusion).')
print('5. Regime-aware models : Separate model for crisis vs. normal regimes (Markov-switching).')

---
## 3 · Non-Technical Report — Sensitivity to Outliers

**What the data showed us:**
A small number of trading days in the AAPL dataset — driven by earnings surprises,
macro shocks, and the COVID crash — were so extreme that they disproportionately
pulled our model's estimates toward them. When those days were given equal weight
alongside all other observations, the model's view of "normal" conditions was
distorted by events that may never repeat in the same form.

**Recommended course of action:**
Rather than letting extreme days fully dominate the model calibration, the team
should use a fitting method that automatically reduces the influence of days whose
returns fall far outside the typical range. Alternatively, the most extreme observations
can be capped at a reasonable threshold before modeling, ensuring the model reflects
recurring market conditions rather than once-a-decade events.

**Factors that impact the portfolio:**
A model overly influenced by outlier days will produce volatility forecasts that
remain elevated long after a crisis has passed, or spike dramatically at the first
hint of turbulence. For a derivatives desk, this means options will be priced too
expensively in calm markets (losing business to competitors) and may still be
mispriced in the direction of the last crisis rather than the current one. Hedging
positions will also be over-sized, tying up margin capital unnecessarily.

---

---
---
# CHALLENGE 4 — OVERFITTING
---

## 4-A · Definition

A model **overfits** when it captures in-sample noise rather than the underlying data-generating process. Formally, the **expected prediction error** decomposes as:

$$\mathbb{E}\left[(y^* - \hat{f}(x^*))^2\right] = \underbrace{\text{Bias}^2[\hat{f}]}_\text{underfitting} + \underbrace{\text{Var}[\hat{f}]}_\text{overfitting} + \sigma^2_\varepsilon$$

Overfitting raises the **variance** term. Model complexity is penalized by information criteria:

$$\text{AIC} = -2\ell(\hat{\theta}) + 2k \qquad \text{BIC} = -2\ell(\hat{\theta}) + k\ln(T)$$

where $\ell$ is the maximized log-likelihood, $k$ is the number of parameters, and $T$ is the sample size. Lower AIC/BIC favors parsimony. Cross-validated RMSE provides a model-free overfitting diagnostic:

$$\text{CV-RMSE} = \sqrt{\frac{1}{K}\sum_{k=1}^{K} \text{MSE}_k}$$

## 4-B · Description

Overfitting occurs when a model is tuned too tightly to the historical data it was trained on, memorizing idiosyncratic noise rather than learning generalizable patterns. In financial modeling, an overfit volatility model will appear impressively accurate in-sample but fail catastrophically on new market data — a dangerous outcome when the model drives live derivatives pricing.

## 4-C · Demonstration

In [ ]:
# ── Use Vol_20 (target) and Lag1_Return as base feature
# Demonstrate overfitting via polynomial degree expansion

feat_of = df[['Lag1_Return']].values
target_of = df['Vol_20'].values

X_train, X_test, y_train, y_test = train_test_split(
    feat_of, target_of, test_size=0.3, shuffle=False   # time-series: no shuffle
)

degrees = range(1, 16)
train_rmse_list, test_rmse_list = [], []

for deg in degrees:
    pipe = make_pipeline(
        PolynomialFeatures(deg, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )
    pipe.fit(X_train, y_train)
    train_rmse_list.append(np.sqrt(mean_squared_error(y_train, pipe.predict(X_train))))
    test_rmse_list.append(np.sqrt(mean_squared_error(y_test,  pipe.predict(X_test))))

results_of = pd.DataFrame({
    'Degree'    : list(degrees),
    'Train RMSE': [round(r, 6) for r in train_rmse_list],
    'Test RMSE' : [round(r, 6) for r in test_rmse_list]
})
print(results_of.to_string(index=False))
print(f"\nMinimum Test RMSE at degree = {results_of.loc[results_of['Test RMSE'].idxmin(), 'Degree']}")

## 4-D · Diagram

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Bias-Variance trade-off
axes[0].plot(list(degrees), train_rmse_list, 'o-', color='#2ca02c',
             label='Train RMSE', linewidth=2, markersize=5)
axes[0].plot(list(degrees), test_rmse_list,  's--', color='#d62728',
             label='Test RMSE',  linewidth=2, markersize=5)

best_deg = results_of.loc[results_of['Test RMSE'].idxmin(), 'Degree']
axes[0].axvline(best_deg, color='navy', linestyle=':', lw=1.5,
                label=f'Optimal degree = {best_deg}')
axes[0].set_xlabel('Polynomial Degree (Model Complexity)')
axes[0].set_ylabel('RMSE')
axes[0].set_title('Bias-Variance Trade-off\n(AAPL Volatility Prediction)', fontweight='bold')
axes[0].legend(fontsize=9)

# ── Panel 2: Fitted curves at degree 1, 3, 12
x_plot = np.linspace(feat_of.min(), feat_of.max(), 300).reshape(-1, 1)
colors_deg = {'1':'#2ca02c', '3':'#1f77b4', '12':'#d62728'}
axes[1].scatter(X_train, y_train, s=5, alpha=0.3, color='gray', label='Train data')
for deg, col in zip([1, 3, 12], colors_deg.values()):
    pipe = make_pipeline(PolynomialFeatures(deg, include_bias=False),
                         StandardScaler(), LinearRegression())
    pipe.fit(X_train, y_train)
    axes[1].plot(x_plot, pipe.predict(x_plot), color=col,
                 lw=2, label=f'Degree {deg}')
axes[1].set_xlabel('Lag1 Return')
axes[1].set_ylabel('Annualized Volatility')
axes[1].set_title('Polynomial Fits:\nUnderfitting vs. Overfitting', fontweight='bold')
axes[1].legend(fontsize=9)

# ── Panel 3: AIC / BIC via OLS for increasing number of lags
lag_cols_all = [f'Lag{i}_Ret' for i in range(1, 11)]
aic_vals, bic_vals = [], []
df_tmp = df.copy()
for i in range(1, 11):
    df_tmp[f'Lag{i}_Ret'] = df_tmp['Return'].shift(i)
df_tmp.dropna(inplace=True)

for k in range(1, 11):
    cols = lag_cols_all[:k]
    X_k  = sm.add_constant(df_tmp[cols])
    m_k  = sm.OLS(df_tmp['Vol_20'], X_k).fit()
    aic_vals.append(m_k.aic)
    bic_vals.append(m_k.bic)

axes[2].plot(range(1, 11), aic_vals, 'o-', color='steelblue', label='AIC', lw=2)
axes[2].plot(range(1, 11), bic_vals, 's--', color='darkorange', label='BIC', lw=2)
axes[2].axvline(np.argmin(aic_vals)+1, color='steelblue', linestyle=':', lw=1.3,
                label=f'AIC min @ lag={np.argmin(aic_vals)+1}')
axes[2].axvline(np.argmin(bic_vals)+1, color='darkorange', linestyle=':', lw=1.3,
                label=f'BIC min @ lag={np.argmin(bic_vals)+1}')
axes[2].set_xlabel('Number of Lag Features')
axes[2].set_ylabel('Information Criterion')
axes[2].set_title('AIC / BIC Model Selection\n(Penalizes Overly Complex Models)', fontweight='bold')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('overfitting_diagram.png', bbox_inches='tight')
plt.show()

## 4-E · Diagnosis

In [ ]:
# ── Diagnosis: Cross-validation RMSE
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
print('=== OVERFITTING DIAGNOSIS — 5-Fold Time-Series Cross-Validation ===')
print(f'{"Degree":<10} {"CV Train RMSE":<18} {"CV Test RMSE":<18} {"Gap (Test-Train)"}')
print('-' * 65)

for deg in [1, 3, 6, 10, 15]:
    pipe = make_pipeline(
        PolynomialFeatures(deg, include_bias=False),
        StandardScaler(),
        LinearRegression()
    )
    train_scores, test_scores = [], []
    for tr_idx, te_idx in tscv.split(feat_of):
        pipe.fit(feat_of[tr_idx], target_of[tr_idx])
        train_scores.append(np.sqrt(mean_squared_error(target_of[tr_idx], pipe.predict(feat_of[tr_idx]))))
        test_scores.append(np.sqrt(mean_squared_error(target_of[te_idx],  pipe.predict(feat_of[te_idx]))))
    tr_mean = np.mean(train_scores)
    te_mean = np.mean(test_scores)
    gap = te_mean - tr_mean
    flag = ' ← OVERFITTING' if gap > 0.005 else ''
    print(f'{deg:<10} {tr_mean:<18.6f} {te_mean:<18.6f} {gap:.6f}{flag}')

## 4-F · Damage

An overfit derivatives model creates compounding failures:

- **Out-of-sample collapse**: Exceptional in-sample accuracy (low MSE) translates to large real-money pricing errors the moment new data arrives — the model has learned AAPL's idiosyncratic 2018–2022 path, not general volatility dynamics.
- **Mispriced derivatives**: Option premiums derived from overfit volatility forecasts are systematically wrong; counterparties with better-calibrated models will systematically trade against the desk.
- **Phantom alpha**: A backtest using an overfit model shows stellar performance that cannot be reproduced live — false confidence leads to over-allocation and larger eventual losses.
- **Hedging breakdown**: Delta and gamma computed from a noise-sensitive model oscillate unnaturally, requiring excessive rebalancing and destroying P&L through transaction costs.
- **Model risk**: Regulators and internal risk functions will fail to validate an overfit model, increasing operational and reputational risk.

## 4-G · Directions

In [ ]:
# ── Direction 1: Ridge vs. Lasso vs. OLS on multi-lag feature set
df_reg = df.copy()
for i in range(1, 11):
    df_reg[f'Lag{i}_Ret'] = df_reg['Return'].shift(i)
df_reg.dropna(inplace=True)

lag_features = [f'Lag{i}_Ret' for i in range(1, 11)]
X_reg = df_reg[lag_features].values
y_reg = df_reg['Vol_20'].values

scaler_reg = StandardScaler()
X_reg_s = scaler_reg.fit_transform(X_reg)

X_tr, X_te, y_tr, y_te = train_test_split(X_reg_s, y_reg, test_size=0.3, shuffle=False)

alphas_reg = np.logspace(-4, 2, 60)
ridge_te, lasso_te = [], []

for a in alphas_reg:
    ridge = Ridge(alpha=a).fit(X_tr, y_tr)
    lasso = Lasso(alpha=a, max_iter=5000).fit(X_tr, y_tr)
    ridge_te.append(np.sqrt(mean_squared_error(y_te, ridge.predict(X_te))))
    lasso_te.append(np.sqrt(mean_squared_error(y_te, lasso.predict(X_te))))

ols_te = np.sqrt(mean_squared_error(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(np.log10(alphas_reg), ridge_te, color='steelblue', lw=2, label='Ridge')
axes[0].plot(np.log10(alphas_reg), lasso_te, color='darkorange', lw=2, label='Lasso')
axes[0].axhline(ols_te, color='red', linestyle='--', lw=1.5, label=f'OLS Test RMSE={ols_te:.5f}')
axes[0].set_xlabel('log₁₀(Regularization α)')
axes[0].set_ylabel('Test RMSE')
axes[0].set_title('Direction 1: Regularization Paths\n(Ridge & Lasso Out-of-Sample RMSE)', fontweight='bold')
axes[0].legend(fontsize=9)

# ── Direction 2: Optimal polynomial + AIC summary bar
method_names   = ['OLS\nDeg-1', 'OLS\nDeg-3', 'OLS\nDeg-12', 'Ridge\n(optimal)', 'Lasso\n(optimal)']
method_rmse_te = [
    test_rmse_list[0],
    test_rmse_list[2],
    test_rmse_list[11],
    ridge_te[np.argmin(ridge_te)],
    lasso_te[np.argmin(lasso_te)]
]
bar_colors = ['#2ca02c','#1f77b4','#d62728','#9467bd','#8c564b']
axes[1].bar(method_names, method_rmse_te, color=bar_colors, edgecolor='k', linewidth=0.7)
axes[1].set_ylabel('Out-of-Sample (Test) RMSE')
axes[1].set_title('Direction 2: Model Comparison\n(Lower Test RMSE = Better Generalization)', fontweight='bold')
for i, v in enumerate(method_rmse_te):
    axes[1].text(i, v + 0.0002, f'{v:.5f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('overfitting_remedies.png', bbox_inches='tight')
plt.show()

print('\n── Recommended Directions ──')
print('1. Ridge (L2) Regularization : Penalizes large coefficients; retains all features.')
print('2. Lasso (L1) Regularization : Sparse solution; eliminates irrelevant lag features.')
print('3. Time-Series Cross-Validation : Respects temporal ordering; unbiased generalization estimate.')
print('4. AIC / BIC Model Selection   : Penalizes complexity; selects parsimonious models.')
print('5. Early stopping (ML models)  : Stop training trees/networks before memorizing noise.')

---
## 4 · Non-Technical Report — Overfitting

**What the data showed us:**
When given the freedom to memorize historical patterns in Apple's price data, our
model achieved near-perfect accuracy on past observations but failed substantially on
data it had never seen. The model learned the specific quirks of the 2018–2022 period
rather than discovering genuinely repeatable price dynamics.

**Recommended course of action:**
Model complexity should be deliberately constrained, and all performance claims must
be validated exclusively on data that played no role in the model's construction. Any
model that cannot demonstrate consistent performance across multiple out-of-sample
time windows should be disqualified from live deployment, regardless of how
impressive its historical statistics appear.

**Factors that impact the portfolio:**
An overfit model is most dangerous precisely when the trading environment changes —
which is exactly when accurate modeling is most valuable. A derivatives desk relying
on an overfit volatility forecast will systematically misprice options at market
turning points, experience unexplained hedging residuals, and face challenges during
internal model validation reviews. The financial cost compounds: bad prices attract
sophisticated counterparties who trade against the desk, while internal risk functions
flag the model for remediation, creating operational disruption.

---

---
---
# STEP 4 — CHALLENGE 5: INTERPRETATION
---

## Background

Having addressed multicollinearity, skewness, sensitivity to outliers, and overfitting
in Steps 1–3, this section turns to the fifth challenge identified in the assignment:
**lack of interpretation**. Of the six challenges listed, this is the one most directly
"solved" by the prior work — each remedy applied in Challenges 1–4 also improved the
interpretability of the model. This section formalizes that connection and introduces
dedicated tools for quantifying feature importance in an interpretable way.

---

## 5-A · Definition

A model suffers from **lack of interpretation** when its internal structure does not
allow a practitioner to understand *why* a prediction was made or *which* inputs
drove it. For linear models, interpretability is linked to the stability of estimated
coefficients $\hat{\beta}$. A coefficient is interpretable only when its standard
error $\text{SE}(\hat{\beta}_j)$ is small relative to its magnitude — formally, when
the $t$-statistic

$$t_j = \frac{\hat{\beta}_j}{\text{SE}(\hat{\beta}_j)}$$

is large enough to reject $H_0: \beta_j = 0$ at a chosen significance level.

For non-linear or high-dimensional models, interpretability is measured through
**feature importance** scores or model-agnostic explainability metrics such as
permutation importance:

$$\text{PI}_j = \text{Error}(\hat{f}, X_{\text{permuted}_j}) - \text{Error}(\hat{f}, X)$$

where permuting feature $j$ and measuring the increase in prediction error reveals
how much the model relied on that feature (Breiman, 2001; Molnar, 2022).

---

## 5-B · Description

Lack of interpretation arises when a model produces accurate-looking outputs but
provides no actionable guidance on which risk factors matter most or how they
influence the forecast. In derivatives modeling, interpretability is not merely an
academic preference — regulators, risk managers, and senior traders must be able to
challenge and validate a model's logic, making black-box predictions operationally
unacceptable regardless of their statistical accuracy.

---

In [ ]:
# ═══════════════════════════════════════════════════════════
# CHALLENGE 5 — INTERPRETATION
# Demonstration: Coefficient stability & permutation importance
# ═══════════════════════════════════════════════════════════

from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor

np.random.seed(42)

# ── Feature set used throughout notebook
interp_features = ['SMA_5', 'SMA_10', 'SMA_20', 'EMA_10', 'EMA_20',
                   'Vol_5', 'Momentum_5', 'Lag1_Return', 'Lag2_Return']
X_interp = df[interp_features].values
y_interp = df['Vol_20'].values

scaler_i = StandardScaler()
X_i_s = scaler_i.fit_transform(X_interp)

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_i_s, y_interp, test_size=0.3, shuffle=False)

# ── OLS with ALL features (collinear → uninterpretable coefficients)
X_ols = sm.add_constant(X_tr_i)
ols_full = sm.OLS(y_tr_i, X_ols).fit()

coef_df = pd.DataFrame({
    'Feature'   : ['const'] + interp_features,
    'Coef'      : ols_full.params,
    'Std_Error' : ols_full.bse,
    't_stat'    : ols_full.tvalues,
    'p_value'   : ols_full.pvalues
})
coef_df['Interpretable'] = coef_df['p_value'].apply(
    lambda p: '✔ Yes' if p < 0.05 else '✖ No (noise)')

print('=== OLS COEFFICIENTS — All Features (collinear, uninterpretable) ===')
print(coef_df[['Feature','Coef','Std_Error','t_stat','p_value','Interpretable']].to_string(index=False))

# ── Permutation importance via Random Forest (tree-based, captures nonlinearity)
rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_tr_i, y_tr_i)

perm = permutation_importance(rf, X_te_i, y_te_i,
                               n_repeats=30, random_state=42, scoring='neg_mean_squared_error')

perm_df = pd.DataFrame({
    'Feature'    : interp_features,
    'Importance' : perm.importances_mean,
    'Std'        : perm.importances_std
}).sort_values('Importance', ascending=False)

print('\n=== PERMUTATION IMPORTANCE — Random Forest ===')
print(perm_df.to_string(index=False))
print('\n→ Features with higher importance cause a larger drop in accuracy when shuffled.')
print('  These are the features the model most relies upon for its forecasts.')

In [ ]:
# ── Diagram: 3-panel interpretability visual
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: OLS t-statistics — which coefficients are "readable"?
colors_t = ['#2ca02c' if p < 0.05 else '#d62728'
            for p in coef_df[coef_df['Feature'] != 'const']['p_value']]
t_vals   = coef_df[coef_df['Feature'] != 'const']['t_stat'].values
feats    = coef_df[coef_df['Feature'] != 'const']['Feature'].values

bars = axes[0].barh(feats, np.abs(t_vals), color=colors_t, edgecolor='k', linewidth=0.6)
axes[0].axvline(1.96, color='green', linestyle='--', lw=1.5, label='|t| = 1.96 (5% sig.)')
axes[0].set_xlabel('|t-statistic|')
axes[0].set_title('Diagram 5-1: OLS |t|-Statistics\n'
                  '(Red = not interpretable, Green = significant)',
                  fontweight='bold')
axes[0].legend(fontsize=9)

# Panel 2: Permutation importance bar chart
axes[1].barh(perm_df['Feature'], perm_df['Importance'],
             xerr=perm_df['Std'], color='steelblue',
             edgecolor='k', linewidth=0.6, capsize=4)
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_xlabel('Mean Increase in MSE when Feature is Permuted')
axes[1].set_title('Diagram 5-2: Permutation Importance\n'
                  '(Larger bar = more important to model output)',
                  fontweight='bold')

# Panel 3: Coefficient path under Ridge (interpretability via shrinkage)
alphas_i  = np.logspace(-2, 4, 100)
ridge_coef_paths = []
for a in alphas_i:
    r = Ridge(alpha=a).fit(X_tr_i, y_tr_i)
    ridge_coef_paths.append(r.coef_)
ridge_coef_paths = np.array(ridge_coef_paths)

for idx, feat in enumerate(interp_features):
    axes[2].plot(np.log10(alphas_i), ridge_coef_paths[:, idx], label=feat, lw=1.5)
axes[2].axhline(0, color='k', lw=0.8)
axes[2].set_xlabel('log₁₀(Ridge Penalty α)')
axes[2].set_ylabel('Standardized Coefficient')
axes[2].set_title('Diagram 5-3: Ridge Coefficient Path\n'
                  '(Stabilizes coefficients → improves interpretability)',
                  fontweight='bold')
axes[2].legend(fontsize=8, loc='right')

plt.tight_layout()
plt.savefig('interpretation_diagram.png', bbox_inches='tight')
plt.show()

## 5-E · Diagnosis

| Test | What to Look For | Interpretation |
|------|-----------------|----------------|
| OLS t-statistics | Many $\|t_j\| < 1.96$ despite non-zero coefficients | Coefficients indistinguishable from noise |
| Coefficient sign | Sign contradicts economic logic (e.g., negative on SMA) | Collinearity distorting estimates |
| Permutation importance | Most features show near-zero importance | Model is not actually using most predictors |
| SHAP values | Inconsistent attribution across similar observations | Non-linear model lacking local interpretability |

A model is **practically uninterpretable** when a risk manager cannot explain — in
economic terms — why a specific forecast was produced or which market condition
drove the output. This is a regulatory and operational failure, not merely a
statistical one.

## 5-F · Damage

Lack of interpretability in a volatility model creates the following compounding failures:

- **Regulatory exposure**: Internal Model Approval processes (Basel III/IV, SR 11-7)
  require models to be documented and challenged. An opaque model will not pass
  validation, delaying or blocking deployment.
- **Model risk escalation**: When a model cannot be explained, errors are harder to
  detect before they become costly. A misspecified feature may be inflating vol
  forecasts for months before anyone identifies the cause.
- **Loss of trader trust**: Derivatives traders rely on model outputs to set bid-ask
  spreads and hedge ratios. If the model cannot be interrogated when its output
  appears unusual, traders will override it — eliminating any systematic edge.
- **Cascading hedging errors**: An uninterpretable model that silently weights
  irrelevant features will produce hedges that appear internally consistent but
  diverge from true market sensitivities, accumulating unbooked risk.

*This challenge is materially improved by the remedies applied in Challenges 1–4:*
*removing multicollinearity stabilizes coefficients; regularization concentrates*
*model weight on genuinely important features; robust fitting reduces the influence*
*of noise-generating outliers; and cross-validation prevents phantom complexity.*

**References:** Molnar (2022) provides the definitive treatment of machine-learning
interpretability methods. Breiman (2001) introduced permutation importance as a
model-agnostic diagnostic for feature relevance.

## 5-G · Directions

| Direction | Mechanism | When to Use |
|-----------|-----------|-------------|
| Ridge / Lasso regularization | Shrinks or zeros noisy coefficients → cleaner attribution | High-dimensional, collinear feature sets |
| PCA pre-processing | Orthogonal components → each has unambiguous variance share | When features are highly correlated |
| Permutation importance | Model-agnostic; works for any estimator | Post-fit audit for any model type |
| Partial dependence plots | Shows marginal effect of each feature holding others fixed | Explaining non-linear models to stakeholders |
| Information criteria (AIC/BIC) | Selects parsimonious models with fewer, more defensible features | Model selection stage |

**Interpretation = Accountability.** On a regulated derivatives desk, every model
parameter must be explainable to risk managers, auditors, and regulators. The
workflow is: (1) remove redundant features via PCA or Lasso; (2) confirm remaining
coefficients are statistically significant; (3) validate feature importance with
permutation analysis; (4) document each feature's economic rationale.

---
# Summary Table — Best-Practices Handbook

| Challenge | Key Diagnostic | Threshold | Primary Remedy |
|-----------|---------------|-----------|----------------|
| Multicollinearity | VIF | > 10 (severe) | PCA · Ridge · Drop features |
| Skewness | Jarque-Bera · γ₁ | p < 0.05 · \|γ₁\| > 0.5 | GJR-GARCH · Skewed-t innovations |
| Sensitivity to Outliers | Cook's D · \|e*\| | D > 4/n · \|e*\| > 3 | Huber M-estimator · Winsorization |
| Overfitting | CV Test RMSE gap · AIC/BIC | Test > Train RMSE | Lasso/Ridge · Time-series CV |

---
---
# References

Bollerslev, Tim. "Generalized Autoregressive Conditional Heteroskedasticity."
*Journal of Econometrics*, vol. 31, no. 3, 1986, pp. 307–327.

Breiman, Leo. "Random Forests." *Machine Learning*, vol. 45, no. 1, 2001, pp. 5–32.

Engle, Robert F. "Autoregressive Conditional Heteroscedasticity with Estimates of
the Variance of United Kingdom Inflation." *Econometrica*, vol. 50, no. 4, 1982,
pp. 987–1007.

Hamilton, James D. *Time Series Analysis*. Princeton University Press, 1994.

Hastie, Trevor, Robert Tibshirani, and Jerome Friedman. *The Elements of Statistical
Learning*. 2nd ed., Springer, 2009.

Hull, John C. *Options, Futures, and Other Derivatives*. 11th ed., Pearson, 2022.

Molnar, Christoph. *Interpretable Machine Learning: A Guide for Making Black Box
Models Explainable*. 2nd ed., 2022, christophm.github.io/interpretable-ml-book/.

Tsay, Ruey S. *Analysis of Financial Time Series*. 3rd ed., Wiley, 2010.

---
*Data Source: Yahoo Finance via yfinance API. AAPL daily adjusted prices,
2018-01-01 – 2025-12-31.*